# 과제 - 신경망을 이용한 손글씨 숫자 인식



## 1. 환경설정



In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요 (저장소 클론 후 경로·모듈 로드)
# Colab GPU에서는 로컬 파일을 직접 읽을 수 없으므로 GitHub에서 원하는 브랜치를 clone/checkout합니다.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from getpass import getpass

    git_url = input("GitHub 저장소 URL (예: github.com/USERNAME/mnist-lab.git): ").strip()
    if not git_url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    branch_name = input("브랜치 이름 (기본: jinhyuk): ").strip() or "jinhyuk"
    token = getpass("GitHub Personal Access Token (public 저장소면 Enter): ")

    # URL 마지막 경로를 저장소 폴더명으로 사용합니다. (예: .../mnist-lab.git -> mnist-lab)
    repo_name = Path(git_url.rstrip("/")).name
    if repo_name.endswith(".git"):
        repo_name = repo_name[:-4]
    repo_path = Path("/content") / repo_name
    os.chdir("/content")

    clone_url = f"https://{token}@{git_url}" if token else f"https://{git_url}"

    if repo_path.exists():
        os.chdir(repo_path)
        !git fetch origin
        !git checkout {branch_name}
        !git pull origin {branch_name}
    else:
        !git clone -b {branch_name} {clone_url}
        os.chdir(repo_path)

    if str(Path.cwd() / "src") not in sys.path:
        sys.path.insert(0, str(Path.cwd() / "src"))
else:
    if "./src" not in sys.path:
        sys.path.insert(0, "./src")

for module_name in ["data", "activations", "layers", "losses", "optimizers", "network", "training"]:
    sys.modules.pop(module_name, None)

print("Working directory:", Path.cwd())
print("Source path:", Path.cwd() / "src")


## 2. 데이터 로드

In [ ]:
from data import load_mnist

(x_train, y_train), (x_val, y_val), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Validation:', x_val.shape, y_val.shape)
print('Test:', x_test.shape, y_test.shape)

## 3. 구현 및 테스트 통과 확인

`src/` 아래 역할별 파일의 **TODO**를 순서대로 구현한 뒤 아래 셀을 실행하세요.
- 주요 구현 파일: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- 구현 파일은 역할별 모듈을 직접 import합니다. 예: `from network import NeuralNetwork`
- 개발 순서: 과제 안내문 참조
- 테스트: `tests/` 아래의 단계별 단위 테스트를 필요한 파일부터 실행합니다. 처음에는 전체 테스트보다 맡은 부분의 테스트 파일을 먼저 실행하세요.
    - ReLU만 확인: `TEST_TARGET = "tests/test_relu.py"`
    - 파일 안의 일부 테스트만 확인: `PYTEST_KEYWORD = "backward"`
    - 전체 테스트 확인: `TEST_TARGET = "tests/"`

In [ ]:
import subprocess
import sys
from pathlib import Path

# Colab/로컬 모두 현재 노트북 실행 위치를 저장소 루트로 사용합니다.
repo_dir = Path.cwd()

# 처음에는 자신이 구현 중인 부분의 테스트 파일만 실행하세요.
# 예: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_relu.py"

# 특정 이름이 들어간 테스트만 실행하고 싶을 때 사용합니다.
# 예: "backward". 전체 파일을 실행하려면 빈 문자열로 둡니다.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("실행 경로:", repo_dir)
print("실행 명령:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\n선택한 테스트를 통과했습니다.")
else:
    print("\n선택한 테스트 중 실패가 있습니다.")


## 4. 하이퍼파라미터 탐색

In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train, evaluate
import numpy as np

hidden_size_candidates = [
    (512,),
    (1024,),
    (256, 128),
    (512, 256),
    (768, 384),
    (1024, 512),
    (768, 512, 256),
    (1024, 512, 256),
    (1024, 768, 384),
    (1024, 768, 512, 256),
]

candidate_pool = [
    {
        "lr": lr,
        "dropout_ratio": dropout_ratio,
        "batch_size": batch_size,
        "hidden_sizes": hidden_sizes,
        "num_hidden_layers": len(hidden_sizes),
    }
    for lr in [0.003, 0.001, 0.0007, 0.0005, 0.0003]
    for dropout_ratio in [0.2, 0.3, 0.4, 0.5]
    for batch_size in [64, 128, 256]
    for hidden_sizes in hidden_size_candidates
]

max_trials = 80
rng = np.random.default_rng(42)
trial_indices = rng.choice(len(candidate_pool), size=min(max_trials, len(candidate_pool)), replace=False)
search_space = [candidate_pool[i] for i in trial_indices]

tuning_epochs = 20
tuning_train_size = min(12000, len(x_train))
tuning_indices = np.random.permutation(len(x_train))[:tuning_train_size]
x_tune = x_train[tuning_indices]
y_tune = y_train[tuning_indices]

best_config = None
best_val_acc = -1
tuning_results = []

print(f"Total candidates: {len(candidate_pool)}")
print(f"Running trials: {len(search_space)}")

for trial, config in enumerate(search_space, start=1):
    print(f"\nTrial {trial}/{len(search_space)}: {config}")
    model = NeuralNetwork(
        use_batchnorm=True,
        use_dropout=True,
        dropout_ratio=config["dropout_ratio"],
        hidden_sizes=config["hidden_sizes"],
    )
    optimizer = Adam(lr=config["lr"])
    history = train(
        model,
        optimizer,
        x_tune,
        y_tune,
        epochs=tuning_epochs,
        batch_size=config["batch_size"],
    )
    val_acc, n_params = evaluate(model, x_val, y_val)
    result = {
        **config,
        "val_acc": float(val_acc),
        "n_params": int(n_params),
        "loss": float(history[-1]),
        "loss_history": [float(loss) for loss in history],
    }
    tuning_results.append(result)
    print(result)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_config = config

tuning_results = sorted(tuning_results, key=lambda x: x["val_acc"], reverse=True)

print("\nTop 5 configs")
for rank, result in enumerate(tuning_results[:5], start=1):
    print(rank, result)

print("\nBest config:", best_config)
print(f"Best validation accuracy: {best_val_acc:.2f}%")

## 4.1 하이퍼파라미터 탐색 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

if not tuning_results:
    raise ValueError("먼저 하이퍼파라미터 탐색 셀을 실행하세요.")

results = sorted(tuning_results, key=lambda x: x["val_acc"], reverse=True)
best_result = results[0]
layer_counts = sorted({result["num_hidden_layers"] for result in results})

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1) 모델 크기와 검증 정확도: 계층 수별 분포
ax = axes[0, 0]
for layer_count in layer_counts:
    group = [result for result in results if result["num_hidden_layers"] == layer_count]
    ax.scatter(
        [result["n_params"] / 1_000_000 for result in group],
        [result["val_acc"] for result in group],
        s=70,
        alpha=0.8,
        label=f"{layer_count} hidden layers",
    )
ax.scatter(
    best_result["n_params"] / 1_000_000,
    best_result["val_acc"],
    marker="*",
    s=350,
    color="crimson",
    edgecolor="black",
    label="best",
)
ax.set_title("Validation Accuracy by Model Size and Depth")
ax.set_xlabel("Parameters (millions)")
ax.set_ylabel("Validation Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()

# 2) 상위 10개 설정 비교
ax = axes[0, 1]
top_results = results[:10]
labels = [
    f"{result['hidden_sizes']}\nlr={result['lr']}, do={result['dropout_ratio']}, bs={result['batch_size']}"
    for result in top_results
]
scores = [result["val_acc"] for result in top_results]
positions = np.arange(len(top_results))
ax.barh(positions, scores, color="steelblue")
ax.set_yticks(positions)
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlim(max(0, min(scores) - 1), min(100, max(scores) + 0.5))
ax.set_title("Top 10 Hyperparameter Configs")
ax.set_xlabel("Validation Accuracy (%)")
ax.grid(True, axis="x", alpha=0.3)

# 3) 계층 수별 성능 분포
ax = axes[1, 0]
layer_acc_groups = [
    [result["val_acc"] for result in results if result["num_hidden_layers"] == layer_count]
    for layer_count in layer_counts
]
ax.boxplot(layer_acc_groups, labels=[str(layer_count) for layer_count in layer_counts], showmeans=True)
layer_best_scores = [max(group) for group in layer_acc_groups]
ax.plot(range(1, len(layer_counts) + 1), layer_best_scores, color="crimson", marker="o", label="best per depth")
ax.set_title("Validation Accuracy Distribution by Hidden Layer Count")
ax.set_xlabel("Number of Hidden Layers")
ax.set_ylabel("Validation Accuracy (%)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()

# 4) 상위 5개 설정의 loss curve
ax = axes[1, 1]
for result in top_results[:5]:
    label = f"{result['hidden_sizes']} / {result['val_acc']:.2f}%"
    ax.plot(result["loss_history"], marker="o", linewidth=2, label=label)
ax.set_title("Loss Curves of Top 5 Configs")
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("hyperparameter_search_summary.png", dpi=200, bbox_inches="tight")
plt.show()

print("Best config:", best_result)


## 4.5 최적 하이퍼파라미터로 최종 학습

In [ ]:
model = NeuralNetwork(
    use_batchnorm=True,
    use_dropout=True,
    dropout_ratio=best_config["dropout_ratio"],
    hidden_sizes=best_config["hidden_sizes"],
)
optimizer = Adam(lr=best_config["lr"])

loss_history = train(
    model,
    optimizer,
    x_train,
    y_train,
    epochs=20,
    batch_size=best_config["batch_size"],
)

## 5. 평가 및 손실 커브

In [ ]:
from training import evaluate, plot_loss_history

val_acc, n_params = evaluate(model, x_val, y_val)
test_acc, _ = evaluate(model, x_test, y_test)

print(f'Validation Accuracy: {val_acc:.2f}%')
print(f'Test Accuracy: {test_acc:.2f}%')
print(f'Total Params: {n_params:,}')

plot_loss_history(loss_history)